## In this approach, we check whether the held-out gene appears in the gene associations (gene space) of the most similar diseases. In this case, we first identify the top 10 most similar disease pairs. Then, for each disease in these pairs, we examine whether the held-out gene exists in the gene space of its top 10 most similar diseases.

### Similarity Calculation for disease(Find Top n Similar Diseases per Disease)

In [41]:
# Cell 1: Find Top n Similar Diseases per Disease
import pandas as pd
import pickle
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

# --- Parameters ---
k_min_assoc = 5   # Parameter k: Only diseases with > k associations
n_neighbors = 10  # Parameter n: Number of similar diseases to consider
vector_type = 'gene' # 'gene', 'hpo', or 'combined'

# --- File Paths ---
gene_embeddings_path = '/home/abamini/PhenoGnet/Code/wandb/latest-run/files/gene_embedding.pkl'
hpo_embeddings_path = '/home/abamini/PhenoGnet/Code/wandb/latest-run/files/hpo_embedding.pkl'
dis2gene_path = '/home/abamini/PhenoGnet/data/processed/dis2g.txt'
hpo2dis_path = '/home/abamini/PhenoGnet/data/processed/hpo2dis.txt'
dis2id_path = '/home/abamini/PhenoGnet/data/processed/dis2id.txt'
gene2id_path = '/home/abamini/PhenoGnet/data/processed/gene2id.txt'
held_out_genes_path = '/home/abamini/PhenoGnet/Code/held_out_genes.txt'
top_sim_file         = 'top_similar_diseases.pkl'
# --- 1. Load Data (Keep as is) ---
with open(gene_embeddings_path, 'rb') as f:
    gene_embeddings = np.asarray(pickle.load(f))
with open(hpo_embeddings_path, 'rb') as f:
    hpo_embeddings = np.asarray(pickle.load(f))

id2umls = {}
with open(dis2id_path) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            id2umls[int(parts[1])] = parts[0]

index2ncbi = {}
with open(gene2id_path) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            index2ncbi[int(parts[1])] = parts[0]

d2g = defaultdict(list)
with open(dis2gene_path) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            did = int(parts[0])
            genes = [int(g) for g in parts[1].split(',') if g]
            d2g[did].extend(genes)
d2g = dict(d2g)

d2h = defaultdict(list)
with open(hpo2dis_path) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            did = int(parts[1])
            hpo_id = [int(h) for h in parts[0].split(',') if h]
            d2h[did].extend(hpo_id)
d2h = dict(d2h)

held_out_dict = {}
try:
    with open(held_out_genes_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                did, gid = int(parts[0]), int(parts[1])
                if gid != -1:
                    held_out_dict[did] = gid
except FileNotFoundError:
    print(f"Warning: {held_out_genes_path} not found.")


filtered_diseases = [
    did for did in (set(d2g.keys()) & set(d2h.keys()))
    if len(set(d2g[did])) > k_min_assoc and len(set(d2h[did])) > k_min_assoc
]
filtered_diseases.sort()

disease_vectors = []
for did in filtered_diseases:
    curr_genes = d2g[did]
    if did in held_out_dict:
        curr_genes = [g for g in curr_genes if g != held_out_dict[did]]
    
    g_vec = np.mean(gene_embeddings[curr_genes], axis=0) if curr_genes else np.zeros(gene_embeddings.shape[1])
    h_vec = np.mean(hpo_embeddings[d2h[did]], axis=0)
    
    if vector_type == 'gene': vec = g_vec
    elif vector_type == 'hpo': vec = h_vec
    #else: vec = np.concatenate([0.9*g_vec, 0.1*h_vec])
    else: vec = (1.5*g_vec + 0.1*h_vec) # we tuned this weights
    disease_vectors.append(vec)

disease_vectors = np.array(disease_vectors)
sim_matrix = cosine_similarity(disease_vectors)
np.fill_diagonal(sim_matrix, -1)

disease_neighbors = {}
for idx, did in enumerate(filtered_diseases):
    sim_scores = sim_matrix[idx]
    # Now you can just take the top N without worrying about the -1 index
    neighbor_indices = np.argsort(sim_scores)[-n_neighbors:][::-1]
    disease_neighbors[did] = [(filtered_diseases[ni], sim_scores[ni]) for ni in neighbor_indices]

with open(top_sim_file, 'wb') as f:
    pickle.dump(disease_neighbors, f)

print(f"Cell 1 Complete: Processed {len(filtered_diseases)} diseases.")

Cell 1 Complete: Processed 1466 diseases.


### For top 10 disease pairs find the occurence of held out gene in it's n most similar disease's gene space

In [42]:
# --- 3. Calculate Similarity for Top 10 Global Pairs ---
sim_matrix = cosine_similarity(disease_vectors)
iu = np.triu_indices(len(filtered_diseases), k=1)
flat_sims = sim_matrix[iu]
top_indices = np.argsort(flat_sims)[-10:][::-1]


def find_gene_depth(target_disease, neighbor_list):
    if target_disease not in held_out_dict:
        return None
    target_gene = held_out_dict[target_disease]
    for rank, neighbor_item in enumerate(neighbor_list, 1):
        # Extract ID if neighbor_list is list of tuples (id, score)
        nid = neighbor_item[0] if isinstance(neighbor_item, (tuple, list, np.ndarray)) else neighbor_item
        if target_gene in d2g.get(nid, []):
            return rank
    return None

def format_depth(depth):
    return f"Rank {depth}" if depth else "Not found"

with open(top_sim_file, 'rb') as f:
    top_neighbors_lookup = pickle.load(f)

print(f"\nTOP 10 SIMILAR PAIRS & RECURSIVE SEARCH (Vector: {vector_type})")
print("=" * 150)
print(f"{'Disease 1':<15} | {'Held-Out Gene':<10} | {'Disease 2':<15} | {'Held-Out Gene':<10} | {'Cosine Sim':<10}")
print("-" * 150)

for idx in top_indices:
    i, j = iu[0][idx], iu[1][idx]
    d1, d2 = filtered_diseases[i], filtered_diseases[j]
    u1, u2 = id2umls.get(d1, str(d1)), id2umls.get(d2, str(d2))
    score = flat_sims[idx]

    # Gene names
    g1_ncbi = index2ncbi.get(held_out_dict.get(d1), "N/A")
    g2_ncbi = index2ncbi.get(held_out_dict.get(d2), "N/A")

    # Perform waterfall searches
    depth1 = find_gene_depth(d1, top_neighbors_lookup.get(d1, []))
    depth2 = find_gene_depth(d2, top_neighbors_lookup.get(d2, []))
    # Print Row
    print(f"{u1:<15} | {g1_ncbi:<10} | {u2:<15} | {g2_ncbi:<10} | {score:.4f}")
    
    # Print Descriptive Summary
    desc1 = f"Gene {g1_ncbi} found in {u1}'s {depth1}th similar disease" if depth1 else f"Gene {g1_ncbi} not found for {u1}"
    desc2 = f"Gene {g2_ncbi} found in {u2}'s {depth2}th similar disease" if depth2 else f"Gene {g2_ncbi} not found for {u2}"
    print(f"  > Summary: {desc1} AND {desc2}.")
    print("-" * 150)



TOP 10 SIMILAR PAIRS & RECURSIVE SEARCH (Vector: gene)
Disease 1       | Held-Out Gene | Disease 2       | Held-Out Gene | Cosine Sim
------------------------------------------------------------------------------------------------------------------------------------------------------
C3553676        | 7547       | C1844020        | 7547       | 1.0000
  > Summary: Gene 7547 found in C3553676's 1th similar disease AND Gene 7547 found in C1844020's 1th similar disease.
------------------------------------------------------------------------------------------------------------------------------------------------------
C4014795        | 6584       | C4310768        | 5771       | 1.0000
  > Summary: Gene 6584 found in C4014795's 1th similar disease AND Gene 5771 found in C4310768's 1th similar disease.
------------------------------------------------------------------------------------------------------------------------------------------------------
C0238198        | 4771       | C022063